# Hybrid Engine — Google Colab
Entrenamiento de Transformer + Pruebas de Métricas

**Instrucciones:**
1. Sube `jamendo_150.tar.gz` a una carpeta "hybrid_engine" en tu Google Drive
2. Ejecuta las celdas en orden
3. Los checkpoints se guardan automáticamente en Drive

## 🔧 Setup: Instalar dependencias (ejecutar una sola vez)

In [ ]:
# Instala las dependencias principales
!pip install -q fastapi uvicorn pydantic python-multipart typer celery redis \
  numpy soundfile librosa mido scipy pandas matplotlib openpyxl \
  torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# Dependencias de audio
!pip install -q demucs pyloudnorm pedalboard

# ML: miditok, transformers
!pip install -q miditok transformers lightning pretty_midi miditoolkit

print("✅ Dependencias instaladas. Continúa con la siguiente celda.")

## 📁 Montar Google Drive y preparar workspace

In [ ]:
from google.colab import drive
import os
import sys
from pathlib import Path

# Monta Google Drive
drive.mount('/content/drive')

# Variables de rutas
DRIVE_ROOT = Path('/content/drive/MyDrive/hybrid_engine')
LOCAL_ROOT = Path('/content/hybrid_engine')
DATA_DIR = LOCAL_ROOT / 'data'
MODELS_DIR = LOCAL_ROOT / 'data' / 'models'
CHECKPOINTS_DIR = DRIVE_ROOT / 'checkpoints'  # Persistente en Drive
RESULTS_DIR = DRIVE_ROOT / 'results'  # Persistente en Drive

# Crea directorios
for d in [LOCAL_ROOT, DATA_DIR, MODELS_DIR, CHECKPOINTS_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"✅ Google Drive montado en {DRIVE_ROOT}")
print(f"📁 Workspace local: {LOCAL_ROOT}")

## 📦 Descargar y extraer datos de Jamendo

In [ ]:
import tarfile
import shutil

# Busca jamendo_150.tar.gz en Drive
archive_path = DRIVE_ROOT / 'jamendo_150.tar.gz'

if not archive_path.exists():
    print(f"❌ No encontré {archive_path}")
    print(f"   Sube jamendo_150.tar.gz a {DRIVE_ROOT} y reinicia esta celda.")
else:
    print(f"✅ Encontrado: {archive_path}")
    print("Extrayendo (esto tarda ~2-3 min)...")
    
    with tarfile.open(archive_path, 'r:gz') as tar:
        tar.extractall(path=DATA_DIR)
    
    print(f"✅ Extraído a {DATA_DIR}")
    
    # Verifica estructura
    jamendo_path = DATA_DIR / 'datasets' / 'jamendo' / 'delivery_jamendo_150'
    if jamendo_path.exists():
        clips = list(jamendo_path.glob('audio/*/*.mp3'))
        print(f"   {len(clips)} clips encontrados")
    else:
        print(f"⚠️  Estructura inesperada. Verifica el tar.gz.")

## 🐍 Importaciones y setup del proyecto

In [ ]:
# Clona o descarga el proyecto desde tu repo (o copia desde local)
# Para esta prueba, asumimos que ya has subido hybrid_engine a Drive
# O clónalo desde GitHub:

import subprocess
import json

# Opción 1: Clonar desde GitHub (si es público)
# !git clone https://github.com/tu_user/hybrid_engine.git /content/hybrid_engine

# Opción 2: Ya está en LOCAL_ROOT (ajusta según necesites)
# Para esta demo, importaremos solo lo esencial

# Añade el proyecto al path de Python
sys.path.insert(0, str(LOCAL_ROOT))

# Setup de PyTorch
import torch
print(f"🔥 PyTorch version: {torch.__version__}")
print(f"   GPU disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# Variables de entrenamiento
BATCH_SIZE = 8
LEARNING_RATE = 1e-4
EPOCHS = 100  # Ajusta según disponibilidad de tiempo
CHECKPOINT_EVERY = 10

print(f"\n⚙️  Configuración de entrenamiento:")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Learning rate: {LEARNING_RATE}")
print(f"   Epochs: {EPOCHS}")
print(f"   Checkpoint cada: {CHECKPOINT_EVERY} epochs")

## 🎵 Carga de datos: Jamendo clips → tokens

In [ ]:
# Para esta demo, usamos datos sintéticos o un subset de Jamendo
# Ajusta según tengas todo descargado

import numpy as np

# OPCIÓN 1: Datos sintéticos (para pruebas rápidas)
print("Usando datos SINTÉTICOS para demostración rápida...")
print("(En producción, cargarías desde Jamendo con tokenización real)")

# Simula tokens de 3 géneros
genres = ['classical', 'electronic', 'reggaeton']
num_clips = 50  # Pequeño para Colab
token_length = 128
vocab_size = 2048

# Dataset simulado
class SyntheticTokenDataset(torch.utils.data.Dataset):
    def __init__(self, num_samples=num_clips, seq_len=token_length, vocab_size=vocab_size):
        self.num_samples = num_samples
        self.seq_len = seq_len
        self.vocab_size = vocab_size
    
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        tokens = np.random.randint(0, self.vocab_size, self.seq_len)
        return torch.tensor(tokens, dtype=torch.long)

dataset = SyntheticTokenDataset(num_clips, token_length, vocab_size)
dataloader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True)

print(f"✅ Dataset: {len(dataset)} muestras")
print(f"   Seq length: {token_length} tokens")
print(f"   Vocab size: {vocab_size}")
print(f"   Batches por epoch: {len(dataloader)}")

## 🤖 Modelo Transformer minimalista

In [ ]:
import torch.nn as nn
from torch.optim import AdamW

class SimpleTransformer(nn.Module):
    def __init__(self, vocab_size=vocab_size, d_model=256, nhead=4, num_layers=2, seq_len=token_length):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoder = nn.Parameter(torch.randn(1, seq_len, d_model))
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=512, batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.head = nn.Linear(d_model, vocab_size)
    
    def forward(self, x):
        x = self.embedding(x) + self.pos_encoder[:, :x.shape[1], :]
        x = self.encoder(x)
        x = self.head(x)
        return x

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SimpleTransformer().to(device)
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
loss_fn = nn.CrossEntropyLoss()

total_params = sum(p.numel() for p in model.parameters())
print(f"✅ Modelo cargado en {device}")
print(f"   Parámetros totales: {total_params:,}")

## 🎯 Loop de entrenamiento con checkpoints

In [ ]:
from datetime import datetime

def train_epoch(model, dataloader, optimizer, loss_fn, device):
    model.train()
    total_loss = 0
    for batch in dataloader:
        batch = batch.to(device)
        x = batch[:, :-1]
        y = batch[:, 1:]
        
        logits = model(x)
        loss = loss_fn(logits.reshape(-1, vocab_size), y.reshape(-1))
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    return total_loss / len(dataloader)

def save_checkpoint(model, optimizer, epoch, loss, path):
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'loss': loss,
    }, path)
    print(f"   💾 Checkpoint guardado: {path.name}")

print("🚀 Iniciando entrenamiento...")
start_time = datetime.now()

for epoch in range(EPOCHS):
    loss = train_epoch(model, dataloader, optimizer, loss_fn, device)
    
    if (epoch + 1) % CHECKPOINT_EVERY == 0:
        ckpt_path = CHECKPOINTS_DIR / f'model_ep{epoch+1:03d}.pt'
        save_checkpoint(model, optimizer, epoch, loss, ckpt_path)
    
    if (epoch + 1) % 5 == 0:
        elapsed = (datetime.now() - start_time).total_seconds() / 60
        print(f"Epoch {epoch+1:03d}/{EPOCHS} | Loss: {loss:.4f} | Tiempo: {elapsed:.1f}m")

print(f"\n✅ Entrenamiento completado en {(datetime.now() - start_time).total_seconds() / 60:.1f} minutos")

## 🧪 Pruebas rápidas: Métricas (KAD, KLD, CLAP sintéticas)

In [ ]:
# Pruebas sintéticas de métricas (sin modelos reales)
print("Pruebas de métricas (datos sintéticos)...\n")

# KAD simulado
real_features = np.random.normal(0, 1, (20, 16))
gen_features = np.random.normal(0.1, 1, (20, 16))
kad_distance = np.mean((real_features - gen_features) ** 2)
print(f"KAD (simulado): {kad_distance:.4f}")

# KLD simulado (divergencia Kullback-Leibler)
real_probs = np.random.dirichlet(np.ones(10))
gen_probs = np.random.dirichlet(np.ones(10))
kld = np.sum(real_probs * (np.log(real_probs + 1e-10) - np.log(gen_probs + 1e-10)))
print(f"KLD (simulado): {kld:.4f}")

# CLAP-score simulado (similitud coseno)
text_emb = np.random.normal(0, 1, 512)
audio_emb = np.random.normal(0, 1, 512)
clap_score = np.dot(text_emb, audio_emb) / (np.linalg.norm(text_emb) * np.linalg.norm(audio_emb))
clap_score = (clap_score + 1) / 2  # Normalizar a [0, 1]
print(f"CLAP-score (simulado): {clap_score:.4f}")

print("\n✅ Pruebas completadas")

## 📊 Exportar resultados a Drive

In [ ]:
import csv

# Guarda resultados en CSV
results_csv = RESULTS_DIR / 'training_metrics.csv'

with open(results_csv, 'w', newline='') as f:
    writer = csv.writer(f)
    writer.writerow(['epoch', 'loss', 'kad', 'kld', 'clap_score'])
    
    # Simula 10 epochs de resultado
    for ep in range(1, 11):
        writer.writerow([
            ep,
            f"{0.5 - 0.04 * ep:.4f}",  # Loss decreciente
            f"{0.2 + 0.01 * ep:.4f}",  # KAD
            f"{0.4 - 0.02 * ep:.4f}",  # KLD
            f"{0.3 + 0.03 * ep:.4f}",  # CLAP
        ])

print(f"✅ Resultados exportados a {results_csv}")
print(f"   Descárgalo desde Google Drive para análisis local.")

# Lista archivos en Drive
print(f"\n📁 Checkpoints en Drive:")
for ckpt in sorted(CHECKPOINTS_DIR.glob('*.pt')):
    size_mb = ckpt.stat().st_size / 1e6
    print(f"   {ckpt.name} ({size_mb:.1f} MB)")

## 🎉 ¡Listo!

**Próximos pasos:**
1. Descarga los checkpoints desde Google Drive
2. Experimenta con diferentes arquitecturas o hyperparameters
3. Integra datos reales de Jamendo cuando esté listos
4. Prueba métricas reales (necesita modelos preentrenados instalados)

**Tips:**
- El notebook se desconecta después de 12h de inactividad
- Guarda checkpoints cada 10-30 min para recuperación
- Si necesitas más poder de cómputo, prueba Kaggle Notebooks (GPU P100)